In [1]:
import psutil
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd
import random
import os
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

from tqdm.notebook import tqdm
import seaborn as sns
from collections import Counter

from glob import glob
import psi4
from helper_CC_ML_spacial import *

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline

  Threads set to 12 by Python driver.


In [2]:
# Most important features (top 5 by SHAP):
# doublecheck: Numerator of the MP2 t2-amplitude, two-electron integral <ik || ab>
# t2start: Initial MP2 t2-amplitude
# t2mag: Magnitude of the MP2 t2-amplitude
# orbdiff: Denominator of the MP2 t2-amplitude
# diag: Binary feature denoting whether a=b (virtual orbits are the same)
top5 = ['doublecheck', 't2start', 't2mag', 'orbdiff', 'diag']

# 31 Features order in X, including t2
properties=['Evir1', 'Hvir1', 'Jvir1', 'Kvir1', 'Evir2', 'Hvir2', 'Jvir2', 'Kvir2', 'Eocc1', 'Jocc1', 'Kocc1', 'Hocc1','Eocc2', 'Jocc2', 'Kocc2', 'Hocc2', 'Jia1', 'Jia2', 'Kia1', 'Kia2','diag', 'orbdiff', 'doublecheck', 't2start', 't2mag', 't2sign', 'Jia1mag', 'Jia2mag','Kia1mag', 'Kia2mag','t2']

In [3]:
basis_sets = ['STO-3G', 'cc-pVDZ', 'aug-cc-pVDZ']

In [4]:
molecules = ["water", "methanol", "ethylene", "ethane", "methane", "ammonia", "formaldehyde"]

In [5]:
random.seed(0)
all_sampled_files = []
for mol in molecules:
    all_files = sorted(glob(os.path.join("data", f"{mol}*.xyz")))
    sample_size = min(100, len(all_files))
    sampled_files = random.sample(all_files, sample_size)

    all_sampled_files.extend(sampled_files)  # append sampled files to a list

    print(f"Total number of {mol} files: {len(all_files)}")
    print(f"{mol}: {len(sampled_files)} files sampled")
    for f in sampled_files:
        print(f)
    print("\n")

Total number of water files: 201
water: 100 files sampled
data/water189.xyz
data/water93.xyz
data/water197.xyz
data/water109.xyz
data/water16.xyz
data/water35.xyz
data/water3.xyz
data/water193.xyz
data/water17.xyz
data/water28.xyz
data/water182.xyz
data/water52.xyz
data/water15.xyz
data/water34.xyz
data/water131.xyz
data/water165.xyz
data/water86.xyz
data/water121.xyz
data/water60.xyz
data/water158.xyz
data/water40.xyz
data/water57.xyz
data/water133.xyz
data/water171.xyz
data/water122.xyz
data/water116.xyz
data/water176.xyz
data/water26.xyz
data/water47.xyz
data/water77.xyz
data/water181.xyz
data/water20.xyz
data/water172.xyz
data/water59.xyz
data/water65.xyz
data/water147.xyz
data/water45.xyz
data/water90.xyz
data/water201.xyz
data/water38.xyz
data/water95.xyz
data/water113.xyz
data/water44.xyz
data/water102.xyz
data/water120.xyz
data/water192.xyz
data/water10.xyz
data/water31.xyz
data/water177.xyz
data/water156.xyz
data/water175.xyz
data/water114.xyz
data/water143.xyz
data/water49.xy

In [6]:
all_sampled_files[:5]

['data/water189.xyz',
 'data/water93.xyz',
 'data/water197.xyz',
 'data/water109.xyz',
 'data/water16.xyz']

In [7]:
random.seed(0)
train_size = min(100, len(all_sampled_files))
train = random.sample(all_sampled_files, train_size)

print(f"Total sampled files: {len(all_sampled_files)}")
print(f"Train set size: {len(train)}")

Total sampled files: 698
Train set size: 100


In [8]:
random.seed(0)
remaining_files = list(set(all_sampled_files) - set(train))
test_size = min(50, len(remaining_files))
test = random.sample(remaining_files, test_size)

print(f"Total sampled files: {len(all_sampled_files)}")
print(f"Train set size: {len(train)}")
print(f"Test set size: {len(test)}")

Total sampled files: 698
Train set size: 100
Test set size: 50


In [9]:
train[:5], test[:5]

(['data/methanol64.xyz',
  'data/methanol52.xyz',
  'data/ethylene42.xyz',
  'data/ethane172.xyz',
  'data/ammonia56.xyz'],
 ['data/ethylene31.xyz',
  'data/methane9.xyz',
  'data/methane23.xyz',
  'data/formaldehyde133.xyz',
  'data/water155.xyz'])

## Note: we do X_test first because we do not adjust the size of this, we test each trained model on the same test set

# X_test

In [10]:
X_test_all = {}
y_test_all = {}

for basis in basis_sets[:2]: 
    filenames = test.copy()
    t1 = time.time()
    
    print(basis)

    data_dict = {}
    for fn in filenames:
        struct = os.path.basename(fn)
        print(f"Processing {struct}")
        
        with open(fn,'r') as f:
            text=f.read()
        
        mol = psi4.geometry(text)
        
        psi4.core.clean()
        psi4.core.be_quiet()
        
        psi4.set_options({'basis': basis,
                          'scf_type': 'pk',
                          'reference': 'rohf',
                          'mp2_type': 'conv',
                          'e_convergence': 1e-8,
                          'd_convergence': 1e-8})

        try:
            rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
            scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
            
            A = HelperCCEnergy(mol, rhf_e, scf_wfn, freeze_core=False)

            MP2T2=A.t2start                                                            # CAN YOU LET ME KNOW IF THIS IS CORRECT
            A.t1 = np.zeros((A.t1.shape))
            A.t2 = MP2T2
            
            MP2E = A.compute_energy(iterate=False)                    # MP2 (initial) energy
            CCSDE = A.compute_energy()                                   # exact CCSD energy

            data = pd.DataFrame(np.array([getattr(A, attr).flatten() for attr in properties]).T, columns=properties)
            data_dict[struct.split('_')[0]] = data

        except Exception as e:
            print(f"Molecule with filename {fn} failed: {e}")
            pass   

    X_test_all[basis] = np.vstack([df[top5].to_numpy() for df in data_dict.values()])
    y_test_all[basis] = np.concatenate([df["t2"].to_numpy().reshape(-1) for df in data_dict.values()])
    t2 = time.time()

    print(f"Time taken for {basis}: {t2-t1} seconds \n")

STO-3G
Processing ethylene31.xyz
Computing RHF reference.


/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 14 basis functions.
(14, 14)
(14, 14)
Building initial guess...

..initialized CCSD in 0.157 seconds.

CCSD Iteration   0: CCSD correlation = -0.123962959050330   dE =  1.23963E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.123962959050330   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.003 seconds!
CCSD Iteration   0: CCSD correlation = -0.123962959050330   dE =  1.23963E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.147170095061054   dE = -2.32071E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.155396106174923   dE = -8.22601E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.161404644719472   dE = -6.00854E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.162395368541487   dE = -9.90724E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.162487973231480   dE = -9.26047E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.16246822352911

/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 9 basis functions.
(9, 9)
(9, 9)
Building initial guess...

..initialized CCSD in 0.307 seconds.

CCSD Iteration   0: CCSD correlation = -0.056652320672720   dE =  5.66523E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.056652320672720   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.001 seconds!
CCSD Iteration   0: CCSD correlation = -0.056652320672720   dE =  5.66523E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.071515095196903   dE = -1.48628E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.076321688000389   dE = -4.80659E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.079015928950109   dE = -2.69424E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.079215207150222   dE = -1.99278E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.079233270537515   dE = -1.80634E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.079232510560152   d

...


Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 24 basis functions.
(24, 24)
(24, 24)
Building initial guess...

..initialized CCSD in 0.183 seconds.

CCSD Iteration   0: CCSD correlation = -0.203582119588782   dE =  2.03582E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.203582119588782   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.001 seconds!
CCSD Iteration   0: CCSD correlation = -0.203582119588782   dE =  2.03582E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.208602265978259   dE = -5.02015E-03   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.211776129283286   dE = -3.17386E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.212657414734564   dE = -8.81285E-04   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.212923243128134   dE = -2.65828E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.212940618417045   dE = -1.73753E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.21294319883689

/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 48 basis functions.
(48, 48)
(48, 48)
Building initial guess...

..initialized CCSD in 1.554 seconds.

CCSD Iteration   0: CCSD correlation = -0.342393165552183   dE =  3.42393E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.342393165552183   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.006 seconds!
CCSD Iteration   0: CCSD correlation = -0.342393165552183   dE =  3.42393E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.357148143475829   dE = -1.47550E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.364515676095851   dE = -7.36753E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.366209975802119   dE = -1.69430E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.366885963132423   dE = -6.75987E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.366947781557595   dE = -6.18184E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.36695647575702

/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 48 basis functions.
(48, 48)
(48, 48)
Building initial guess...

..initialized CCSD in 1.443 seconds.

CCSD Iteration   0: CCSD correlation = -0.342321313683187   dE =  3.42321E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.342321313683187   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.004 seconds!
CCSD Iteration   0: CCSD correlation = -0.342321313683187   dE =  3.42321E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.357095211095332   dE = -1.47739E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.364452366106938   dE = -7.35716E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.366147008538323   dE = -1.69464E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.366820453490645   dE = -6.73445E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.366882084026104   dE = -6.16305E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.36689072590511

# X_train

In [ ]:
sizes = [1, 5, 10, 20, 40, 60, 100]
X_train_all = {}
y_train_all = {}

for basis in basis_sets: 
    for n in sizes:
        filenames = train[:n]
        t1 = time.time()
        
        print(f"{basis} Basis, N = {n} training molecules")
        print(f"Training molecules: {filenames}")

        # get training molecule
        data_dict = {}
        for fn in filenames:
            struct = os.path.basename(fn)
            print(f"Processing {struct}")
            
            with open(fn,'r') as f:
                text=f.read()
            
            mol = psi4.geometry(text)
            
            psi4.core.clean()
            psi4.core.be_quiet()
            
            psi4.set_options({'basis': basis,
                              'scf_type':     'pk',
                              'reference':    'rohf',
                              'mp2_type':     'conv',
                              'e_convergence': 1e-8,
                              'd_convergence': 1e-8})

            try:
                
                rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
                scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
                
                A=HelperCCEnergy(mol, rhf_e, scf_wfn,freeze_core=False)
    
    
                MP2T2=A.t2start                                                            # CAN YOU LET ME KNOW IF THIS IS CORRECT
                A.t1 = np.zeros((A.t1.shape))
                A.t2 = MP2T2
                
                MP2E = A.compute_energy(iterate=False)                    # MP2 (initial) energy
                CCSDE = A.compute_energy()                                   # exact CCSD energy
    
                data=pd.DataFrame(np.array([getattr(A,attr).flatten() for attr in properties]).T,columns=properties)
                data_dict[struct.split('_')[0]]=data
            except Exception as e:
                print(f"Molecule with filename {fn} failed: {e}")
                pass   

        X_train_all[basis] = np.vstack([df[top5].to_numpy() for df in data_dict.values()])
        y_train_all[basis] = np.concatenate([df["t2"].to_numpy().reshape(-1) for df in data_dict.values()])
        
        model = XGBRegressor(
            n_estimators=400,
            max_depth=12,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            reg_alpha=0.0,
            tree_method="hist",
            n_jobs=-1,
            random_state=42
        )

        scaler = MinMaxScaler(feature_range=(-1,1))

        X_train_scaled = scaler.fit_transform(X_train_all[basis])
        X_test_scaled = scaler.transform(X_test_all[basis])

        model.fit(X_train_scaled, y_train_all[basis])
        y_pred = model.predict(X_test_scaled)
        
        r2 = r2_score(y_test_all[basis], y_pred)
        mae = mean_absolute_error(y_test_all[basis], y_pred)
        rmse = root_mean_squared_error(y_test_all[basis], y_pred)
    
        print(f"N: {n}, MAE: {mae}, RMSE: {rmse}, R2: {r2}")
        
        joblib.dump(scaler, f"out/{basis}_scaler_{n}.pkl")
        model.save_model(f"out/{basis}_model_{n}.json")

        print("Saved file, saved scaler.")
        
        print("\n")
    print("\n")

STO-3G Basis, N = 1 training molecules
Training molecules: ['data/methanol64.xyz']
Processing methanol64.xyz
Computing RHF reference.


/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 14 basis functions.
(14, 14)
(14, 14)
Building initial guess...

..initialized CCSD in 0.065 seconds.

CCSD Iteration   0: CCSD correlation = -0.086308762798468   dE =  8.63088E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.086308762798468   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.001 seconds!
CCSD Iteration   0: CCSD correlation = -0.086308762798468   dE =  8.63088E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.106562432483233   dE = -2.02537E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.112966308969718   dE = -6.40388E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.116259693818527   dE = -3.29338E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.116582793929931   dE = -3.23100E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.116666236746932   dE = -8.34428E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.11666748422261

/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 14 basis functions.
(14, 14)
(14, 14)
Building initial guess...

..initialized CCSD in 0.081 seconds.

CCSD Iteration   0: CCSD correlation = -0.086308762798468   dE =  8.63088E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.086308762798468   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.001 seconds!
CCSD Iteration   0: CCSD correlation = -0.086308762798468   dE =  8.63088E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.106562432483233   dE = -2.02537E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.112966308969718   dE = -6.40388E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.116259693818527   dE = -3.29338E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.116582793929931   dE = -3.23100E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.116666236746931   dE = -8.34428E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.11666748422261

...


Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 7 basis functions.
(7, 7)
(7, 7)
Building initial guess...

..initialized CCSD in 0.013 seconds.

CCSD Iteration   0: CCSD correlation = -0.034849534319774   dE =  3.48495E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.034849534319774   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.001 seconds!
CCSD Iteration   0: CCSD correlation = -0.034849534319774   dE =  3.48495E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.044099272203228   dE = -9.24974E-03   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.046973982840840   dE = -2.87471E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.048514398431295   dE = -1.54042E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.048627603404545   dE = -1.13205E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.048634770939612   dE = -7.16754E-06   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.048632669351615   d

/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 16 basis functions.
(16, 16)
(16, 16)
Building initial guess...

..initialized CCSD in 0.166 seconds.

CCSD Iteration   0: CCSD correlation = -0.107488760978257   dE =  1.07489E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.107488760978257   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.001 seconds!
CCSD Iteration   0: CCSD correlation = -0.107488760978257   dE =  1.07489E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.133395090386218   dE = -2.59063E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.141476201439748   dE = -8.08111E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.145945119657337   dE = -4.46892E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.146344133316848   dE = -3.99014E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.146385821686880   dE = -4.16884E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.14638449065057

/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/machine_learning/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 9 basis functions.
(9, 9)
(9, 9)
Building initial guess...

..initialized CCSD in 0.016 seconds.

CCSD Iteration   0: CCSD correlation = -0.056856599896504   dE =  5.68566E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.056856599896504   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.001 seconds!
CCSD Iteration   0: CCSD correlation = -0.056856599896504   dE =  5.68566E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.071774716243614   dE = -1.49181E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.076604324992622   dE = -4.82961E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.079316796366455   dE = -2.71247E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.079517872621735   dE = -2.01076E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.079536264880760   dE = -1.83923E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.079535502376514   d

# Hyperparameter tuning

In [ ]:
# something to do with grid search pipeline here....